In [ ]:
# Xnet

# !pip install transformers datasets scikit-learn -q

In [2]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import inspect

from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from transformers import AutoTokenizer, AutoModel, TrainingArguments, Trainer, set_seed, EarlyStoppingCallback


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = "/content/drive/MyDrive/XNET_MODEL"
os.makedirs(SAVE_PATH, exist_ok=True)

In [ ]:
# Model Settings

MODEL_NAME = "roberta-base"
MAX_LENGTH = 256
NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
LEARNING_RATE = 2e-5
DROPOUT = 0.2
SEED = 42

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Load Data & Preprocessing 

train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")

def clean(x):
    return "" if pd.isna(x) else str(x).strip()

for df in [train_df, val_df, test_df]:
    df["src"] = df["text_src"].apply(clean)
    df["tgt"] = df["text_tgt"].apply(clean)

labels = sorted(train_df["label"].unique())
label2id = {l:i for i,l in enumerate(labels)}
id2label = {i:l for l,i in label2id.items()}

for df in [train_df, val_df, test_df]:
    df["label"] = df["label"].map(label2id)


In [4]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array(list(label2id.values())),
    y=train_df["label"]
)
class_weights = torch.tensor(class_weights, dtype=torch.float)

In [ ]:
# Tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    enc = tokenizer(
        batch["src"],
        batch["tgt"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )
    enc["labels"] = batch["label"]
    return enc

train_ds = Dataset.from_pandas(train_df[["src","tgt","label"]]).map(tokenize, batched=True)
val_ds = Dataset.from_pandas(val_df[["src","tgt","label"]]).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df[["src","tgt","label"]]).map(tokenize, batched=True)

cols = ["input_ids","attention_mask","labels"]
train_ds.set_format("torch", columns=cols)
val_ds.set_format("torch", columns=cols)
test_ds.set_format("torch", columns=cols)

In [ ]:
# Xnet Model 
class Xnet_Net(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size

        # f function
        self.f = nn.Linear(hidden * 4, hidden)
        self.act = nn.Tanh()
        self.dropout = nn.Dropout(0.2)

        self.classifier = nn.Linear(hidden, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None):

        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        hidden_states = outputs.last_hidden_state
        batch_size = input_ids.size(0)

        # Find [SEP] tokens to split src & tgt
        sep_token_id = 2  # RoBERTa </s>

        sep_positions = (input_ids == sep_token_id).nonzero(as_tuple=False)

        o_list = []
        n_list = []

        for i in range(batch_size):
            positions = sep_positions[sep_positions[:, 0] == i][:, 1]

            # positions[0] = first </s> → end of src
            # positions[1] = second </s> → end of tgt

            src_end = positions[0]
            tgt_end = positions[1]

        
            #Extract token-level representations
            # o = last token of SRC
            o = hidden_states[i, src_end - 1, :]

            # n = last token of TGT
            n = hidden_states[i, tgt_end - 1, :]

            o_list.append(o)
            n_list.append(n)

        o = torch.stack(o_list)
        n = torch.stack(n_list)

        # XNET INTERACTION
        interaction = torch.cat([
            o,
            n,
            o - n,
            o * n
        ], dim=1)

        # f → u
        u = self.act(self.f(interaction))
        u = self.dropout(u)

        logits = self.classifier(u)

        loss = None
        if labels is not None:
            loss_fn = nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

model = Xnet_Net(MODEL_NAME, len(labels)).to(device)

In [ ]:
# Metrices

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1,
        "precision": precision,
        "recall": recall
    }


In [ ]:
# Training Args

signature = inspect.signature(TrainingArguments.__init__)
valid_params = set(signature.parameters.keys())
args_dict = {
    "output_dir": "./tmp",
    "num_train_epochs": NUM_EPOCHS,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "report_to": "none",
    "fp16": torch.cuda.is_available()
}

if "evaluation_strategy" in valid_params:
    args_dict["evaluation_strategy"] = "epoch"
elif "eval_strategy" in valid_params:
    args_dict["eval_strategy"] = "epoch"

if "save_strategy" in valid_params:
    args_dict["save_strategy"] = "epoch"

if "logging_strategy" in valid_params:
    args_dict["logging_strategy"] = "epoch"

args = TrainingArguments(**args_dict)


In [ ]:
# Trainer 

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

In [ ]:
# Test

pred = trainer.predict(test_ds)
preds = np.argmax(pred.predictions, axis=1)
true = pred.label_ids
print("\nClassification Report:\n")
print(classification_report(true, preds))
print("\nConfusion Matrix:\n")
print(confusion_matrix(true, preds))

In [ ]:
# Save the Model

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("\nModel saved at:", SAVE_PATH)